In [ ]:
#importando bibliotecas
import tabula
import pandas as pd
import re
import numpy as np

In [ ]:
#BP ATIVOS
dfs = tabula.read_pdf(
    "BP_ATIVOS.pdf",
    pages=2,
    stream=True,
    guess=False,
    area=[60, 30, 830, 800]  # ajuste fino da página
)

df = dfs[0]
df.columns = ["col1","col2","col3"]
df = df.iloc[3:].reset_index(drop=True)
df[["codigo_conta","descricao_conta"]] = df["col1"].str.split(" ",n=1,expand=True)
df = df.rename(columns={"col2":"valor_2025","col3":"valor_2024"})



df_bp_ativos = df[
["codigo_conta", "descricao_conta", "valor_2025", "valor_2024"]
]


In [ ]:
#BP_PASSIVOS
dfs_p = tabula.read_pdf(
    "BP_PASSIVOS.pdf",
    pages=2,
    stream=True,
    guess=False,
    area=[60, 30, 830, 800]  # ajuste fino da página
)
df_p = dfs_p[0]
df_p.columns = ["col1","col2","col3"]
df_p = df_p.iloc[3:].reset_index(drop=True)
df_p[["codigo_conta","descricao_conta"]] = df_p["col1"].str.split(" ",n=1,expand=True)
df_p = df_p.rename(columns={"col2":"valor_2025","col3":"valor_2024"})


df_bp_passivos = df_p[
["codigo_conta", "descricao_conta", "valor_2025", "valor_2024"]
]


In [ ]:
#DRE
dfs_dre = tabula.read_pdf(
    "DRE.pdf",
    pages=2,
    stream=True,
    guess=False,
    area=[60, 30, 830, 800]  # ajuste fino da página
)
df_dre = dfs_dre[0]
df_dre.columns = ["col1","col2","col3"]
df_dre = df_dre.iloc[3:].reset_index(drop=True)
df_dre[["codigo_conta","descricao_conta"]] = df_dre["col1"].str.split(" ",n=1,expand=True)
df_dre = df_dre.rename(columns={"col2":"valor_2025","col3":"valor_2024"})

df_dre = df_dre[
["codigo_conta", "descricao_conta", "valor_2025", "valor_2024"]
]


In [ ]:
#DFC
dfs_dfc = tabula.read_pdf(
    "DFC.pdf",
    pages=2,
    stream=True,
    guess=False,
    area=[60, 30, 830, 800],
    columns=[200,500]  
)
df_dfc = dfs_dfc[0]
df_dfc.columns = ["col1", "col2", "col3"]
df_dfc = df_dfc.iloc[3:].reset_index(drop=True)
df_dfc[["codigo_conta","descricao_conta"]] = df_dfc["col1"].str.split(" ",n=1,expand=True)
df_dfc = df_dfc.rename(columns={"col2":"valor_2025","col3":"valor_2024"})
df_dfc = df_dfc[
["codigo_conta", "descricao_conta", "valor_2025", "valor_2024"]
]
#Correção de colunas
df_dfc["texto_extra"] = df_dfc["valor_2025"].str.extract(r'^([^\d]+)')
df_dfc["valor_2025"] = df_dfc["valor_2025"].str.extract(r'([\d\.\,]+)')
df_dfc["descricao_conta"] = (
    df_dfc["descricao_conta"].fillna("")+""+df_dfc["texto_extra"].fillna("")
).str.strip()
df_dfc = df_dfc.drop(columns=["texto_extra"])



In [ ]:
#transformar strings em float
def para_float(df, coluna):
    df[coluna] = (
        df[coluna]
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", "", regex=False)  # remove vírgula em vez de trocar por ponto
        .str.strip()
        .str.replace("None", "0", regex=False)
        .str.replace("nan", "0", regex=False)  # trata NaN
        .replace("", "0")                       # trata string vazia
    )
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce").fillna(0)
    return df
df_dfc = para_float(df_dfc,"valor_2025")
df_dfc = para_float(df_dfc,"valor_2024")
df_dre = para_float(df_dre,"valor_2025")
df_dre = para_float(df_dre,"valor_2024")
df_bp_ativos = para_float(df_bp_ativos,"valor_2025")
df_bp_ativos = para_float(df_bp_ativos,"valor_2024")
df_bp_passivos = para_float(df_bp_passivos,"valor_2025")
df_bp_passivos = para_float(df_bp_passivos,"valor_2024")



In [ ]:
import numpy as np
#função para buscar valores solicitados 
def pegar_valor(df,codigo,coluna):
    resultado = df[df["codigo_conta"]==codigo][coluna].values
    return resultado[0] if len(resultado) > 0 else 0 

#dre

dados_dre = {
    "descricao": [
        "Receita Líquida",
        "CPV",
        "Lucro Bruto",
        "EBIT (Resultado Operacional)",
        "Lucro Antes do IR",
        "Lucro Líquido",
        "Depreciação e Amortização"
    ],
    "codigo": ["3.01", "3.02", "3.03", "3.05", "3.07", "3.11", "6.01.01.04"],
    "valor_2025": [
        pegar_valor(df_dre, "3.01", "valor_2025"),
        pegar_valor(df_dre, "3.02", "valor_2025"),
        pegar_valor(df_dre, "3.03", "valor_2025"),
        pegar_valor(df_dre, "3.05", "valor_2025"),
        pegar_valor(df_dre, "3.07", "valor_2025"),
        pegar_valor(df_dre, "3.11", "valor_2025"),
        pegar_valor(df_dfc, "6.01.01.04", "valor_2025"),
    ],
    "valor_2024": [
        pegar_valor(df_dre, "3.01", "valor_2024"),
        pegar_valor(df_dre, "3.02", "valor_2024"),
        pegar_valor(df_dre, "3.03", "valor_2024"),
        pegar_valor(df_dre, "3.05", "valor_2024"),
        pegar_valor(df_dre, "3.07", "valor_2024"),
        pegar_valor(df_dre, "3.11", "valor_2024"),
        pegar_valor(df_dfc, "6.01.01.04", "valor_2024"),
    ]
}

df_dre_final = pd.DataFrame(dados_dre)
#add o EBITDA
ebitda_2025 = pegar_valor(df_dre, "3.05", "valor_2025") + pegar_valor(df_dfc, "6.01.01.04", "valor_2025")
ebitda_2024 = pegar_valor(df_dre, "3.05", "valor_2024") + pegar_valor(df_dfc, "6.01.01.04", "valor_2024")
nova_linha = {
    "descricao": "EBITDA",
    "codigo": "calc",
    "valor_2025":ebitda_2025,
    "valor_2024":ebitda_2024
}
df_dre_final = pd.concat([df_dre_final,pd.DataFrame([nova_linha])],ignore_index=True)

In [ ]:

#BP
dados_bp = {
    "descricao": [
        "Ativo Total",
        "Ativo Circulante",
        "Caixa e Equivalentes",
        "Aplicações Financeiras (CP)",
        "Passivo Circulante",
        "Empréstimos CP",
        "Passivo Não Circulante",
        "Empréstimos LP",
        "Patrimônio Líquido"
    ],
    "codigo":["1","1.01","1.01.01","1.01.02",
              "2.01","2.01.04","2.02","2.02.01","2.03"],
    "valor_2025":[
        pegar_valor(df_bp_ativos,"1","valor_2025"),
        pegar_valor(df_bp_ativos,"1.01","valor_2025"),
        pegar_valor(df_bp_ativos,"1.01.01","valor_2025"),
        pegar_valor(df_bp_ativos,"1.01.02","valor_2025"),
        pegar_valor(df_bp_passivos,"2.01","valor_2025"),
        pegar_valor(df_bp_passivos,"2.01.04","valor_2025"),
        pegar_valor(df_bp_passivos,"2.02","valor_2025"),
        pegar_valor(df_bp_passivos,"2.02.01","valor_2025"),
        pegar_valor(df_bp_passivos,"2.03","valor_2025"),
    ],
    "valor_2024":[
        pegar_valor(df_bp_ativos,"1","valor_2024"),
        pegar_valor(df_bp_ativos,"1.01","valor_2024"),
        pegar_valor(df_bp_ativos,"1.01.01","valor_2024"),
        pegar_valor(df_bp_ativos,"1.01.02","valor_2024"),
        pegar_valor(df_bp_passivos,"2.01","valor_2024"),
        pegar_valor(df_bp_passivos,"2.01.04","valor_2024"),
        pegar_valor(df_bp_passivos,"2.02","valor_2024"),
        pegar_valor(df_bp_passivos,"2.02.01","valor_2024"),
        pegar_valor(df_bp_passivos,"2.03","valor_2024")
    ]

}
df_bp_final = pd.DataFrame(dados_bp)


In [ ]:
#cálculo de indicadores
def pegar_valor(df,codigo,coluna):
    resultado = df[df["codigo"]==codigo][coluna].values
    return resultado[0] if len(resultado) > 0 else 0 
#DRE
receita_2025 = pegar_valor(df_dre_final,"3.01", "valor_2025")
receita_2024 = pegar_valor(df_dre_final,"3.01", "valor_2024")

lucro_bruto_2025 = pegar_valor(df_dre_final, "3.03", "valor_2025")
lucro_bruto_2024 = pegar_valor(df_dre_final, "3.03", "valor_2024")

ebit_2025 = pegar_valor(df_dre_final, "3.05", "valor_2025")
ebit_2024 = pegar_valor(df_dre_final, "3.05", "valor_2024")

lucro_liq_2025 = pegar_valor(df_dre_final, "3.11", "valor_2025")
lucro_liq_2024 = pegar_valor(df_dre_final, "3.11", "valor_2024")

ebitda_2025 = pegar_valor(df_dre_final, "calc", "valor_2025")
ebitda_2024 = pegar_valor(df_dre_final, "calc", "valor_2024")

# BP
ativo_total_2025 = pegar_valor(df_bp_final, "1", "valor_2025")
ativo_total_2024 = pegar_valor(df_bp_final, "1", "valor_2024")

ativo_circ_2025 = pegar_valor(df_bp_final, "1.01", "valor_2025")
ativo_circ_2024 = pegar_valor(df_bp_final, "1.01", "valor_2024")

caixa_2025 = pegar_valor(df_bp_final, "1.01.01", "valor_2025")
caixa_2024 = pegar_valor(df_bp_final, "1.01.01", "valor_2024")

aplic_2025 = pegar_valor(df_bp_final, "1.01.02", "valor_2025")
aplic_2024 = pegar_valor(df_bp_final, "1.01.02", "valor_2024")

passivo_circ_2025 = pegar_valor(df_bp_final, "2.01", "valor_2025")
passivo_circ_2024 = pegar_valor(df_bp_final, "2.01", "valor_2024")

emprest_cp_2025 = pegar_valor(df_bp_final, "2.01.04", "valor_2025")
emprest_cp_2024 = pegar_valor(df_bp_final, "2.01.04", "valor_2024")

emprest_lp_2025 = pegar_valor(df_bp_final, "2.02.01", "valor_2025")
emprest_lp_2024 = pegar_valor(df_bp_final, "2.02.01", "valor_2024")

pl_2025 = pegar_valor(df_bp_final, "2.03", "valor_2025")
pl_2024 = pegar_valor(df_bp_final, "2.03", "valor_2024")
#divida total e liquida
divida_total_2025 = emprest_cp_2025 + emprest_lp_2025
divida_total_2024 = emprest_cp_2024 + emprest_lp_2024

divida_liq_2025 = divida_total_2025 - caixa_2025 - aplic_2025
divida_liq_2024 = divida_total_2024 - caixa_2024 - aplic_2024

# Alíquota efetiva de IR
aliq_ir_2025 = abs(pegar_valor(df_dre_final, "3.08", "valor_2025")) / abs(pegar_valor(df_dre_final, "3.07", "valor_2025"))
aliq_ir_2024 = abs(pegar_valor(df_dre_final, "3.08", "valor_2024")) / abs(pegar_valor(df_dre_final, "3.07", "valor_2024"))


In [ ]:
#indicadores
indicadores = {
    "indicador": [
        # Lucratividade
        "Margem Bruta",
        "Margem EBITDA",
        "Margem Operacional",
        "Margem Líquida",
        # Retorno
        "ROE",
        "ROA",
        "ROCE",
        "ROIC",
        # Liquidez
        "Liquidez Corrente",
        "Liquidez Imediata",
        # Alavancagem
        "Debt Ratio",
        "Debt to Equity",
    ],
    "categoria": [
        "Lucratividade", "Lucratividade", "Lucratividade", "Lucratividade",
        "Retorno", "Retorno", "Retorno", "Retorno",
        "Liquidez", "Liquidez",
        "Alavancagem", "Alavancagem",
    ],
    "valor_2025": [
        lucro_bruto_2025  / receita_2025,
        ebitda_2025       / receita_2025,
        ebit_2025         / receita_2025,
        lucro_liq_2025    / receita_2025,

        lucro_liq_2025    / pl_2025,
        lucro_liq_2025    / ativo_total_2025,
        ebit_2025         / (ativo_total_2025 - passivo_circ_2025),
        (ebit_2025 * (1 - aliq_ir_2025)) / (pl_2025 + divida_liq_2025),

        ativo_circ_2025   / passivo_circ_2025,
        (caixa_2025 + aplic_2025) / passivo_circ_2025,

        divida_total_2025 / ativo_total_2025,
        divida_total_2025 / pl_2025,
    ],
    "valor_2024": [
        lucro_bruto_2024  / receita_2024,
        ebitda_2024       / receita_2024,
        ebit_2024         / receita_2024,
        lucro_liq_2024    / receita_2024,

        lucro_liq_2024    / pl_2024,
        lucro_liq_2024    / ativo_total_2024,
        ebit_2024         / (ativo_total_2024 - passivo_circ_2024),
        (ebit_2024 * (1 - aliq_ir_2024)) / (pl_2024 + divida_liq_2024),

        ativo_circ_2024   / passivo_circ_2024,
        (caixa_2024 + aplic_2024) / passivo_circ_2024,

        divida_total_2024 / ativo_total_2024,
        divida_total_2024 / pl_2024,
    ]
}

df_indicadores = pd.DataFrame(indicadores)
cols_pct = df_indicadores["categoria"].isin(["Lucratividade", "Retorno"])
df_indicadores.loc[cols_pct, "valor_2025"] = df_indicadores.loc[cols_pct, "valor_2025"] * 100
df_indicadores.loc[cols_pct, "valor_2024"] = df_indicadores.loc[cols_pct, "valor_2024"] * 100
df_indicadores.to_csv("petrobras.csv",encoding="latin1")